# Example 05 — Geometric Nonlinearity (Hardening FRF)

FRF of a 2-DOF chain of oscillators with cubic geometric stiffening on both DOFs, demonstrating the characteristic hardening jump and frequency-shifted resonance peak. (Krack & Gross 2019)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.linalg import eigh

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.systems.oscillators import ChainOfOscillators
from nlvib.nonlinearities.elements import cubic_spring
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

In [ ]:
# --- System setup ---
MASSES = [1.0, 1.0]
STIFFNESSES = [0.5, 1.0, 0.5]
DAMPINGS = [0.01, 0.01, 0.01]
K3 = 2.0
FORCE_AMP = 0.05
OMEGA_MIN = 0.4
OMEGA_MAX = 1.6
N_HARMONICS = 3

system = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)
system.add_nonlinear_element(cubic_spring(k3=K3, dof_index=0))
system.add_nonlinear_element(cubic_spring(k3=K3, dof_index=1))
excitation = {'dof': 0, 'amplitude': FORCE_AMP, 'harmonic': 1}

K_dense = system.K.toarray(); M_dense = system.M.toarray()
eigvals, _ = eigh(K_dense, M_dense)
omega_linear = float(np.sqrt(np.abs(eigvals[0])))
print(f'Linear 1st natural frequency: {omega_linear:.4f} rad/s')

In [ ]:
# --- Initial solution at OMEGA_MIN ---
n_dof = system.n_dof
n_total = n_dof * (2 * N_HARMONICS + 1)
Q0 = np.zeros(n_total)
for _ in range(30):
    R, J = hb_residual(Q0, OMEGA_MIN, system, N_HARMONICS, excitation)
    if np.linalg.norm(R) < 1e-10: break
    try: dQ = np.linalg.solve(J, -R)
    except np.linalg.LinAlgError: dQ = np.linalg.lstsq(J, -R, rcond=None)[0]
    Q0 += dQ

In [ ]:
# --- Arc-length continuation ---
def residual_fn(Q, omega):
    return hb_residual(Q, omega, system, N_HARMONICS, excitation)

opts = ContinuationOptions(
    ds_initial=0.01, ds_min=1e-6, ds_max=0.05, max_steps=800,
    newton_tol=1e-8, lambda_min=OMEGA_MIN, lambda_max=OMEGA_MAX,
)
result = ContinuationSolver().run(residual_fn, Q0, OMEGA_MIN, opts)
print(f'Continuation: {result.n_steps} steps, {result.message}')

solutions = result.solutions
omegas = solutions[:, -1]
cos1_dof0 = solutions[:, n_dof*1 + 0]
sin1_dof0 = solutions[:, n_dof*2 + 0]
amp_dof0 = np.sqrt(cos1_dof0**2 + sin1_dof0**2)
stability = result.stability

In [ ]:
# --- Save FRF plot ---
if len(amp_dof0) > 0:
    peak_idx = int(np.argmax(amp_dof0))
    peak_amp = float(amp_dof0[peak_idx])
    peak_omega = float(omegas[peak_idx])
else:
    peak_amp = peak_omega = float('nan')
hardening_ratio = peak_omega / omega_linear if omega_linear > 0 else float('nan')

output_dir = Path('..') / 'examples' / '05_geometric_nonlinearity' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
for i in range(len(omegas) - 1):
    is_stable = not bool(stability[i])
    ax.plot(omegas[i:i+2], amp_dof0[i:i+2],
            color='tab:blue' if is_stable else 'tab:red',
            linestyle='-' if is_stable else '--', linewidth=1.5)
handles = [
    Line2D([0], [0], color='tab:blue', linestyle='-', label='stable'),
    Line2D([0], [0], color='tab:red', linestyle='--', label='unstable'),
]
ax.legend(handles=handles)
ax.set_xlabel(r'Excitation frequency $\Omega$ (rad/s)')
ax.set_ylabel(r'Amplitude $|q_0|$ (harmonic 1)')
ax.set_title('Example 05 — Geometric Nonlinearity (hardening FRF)')
ax.axvline(peak_omega, color='gray', linestyle=':', linewidth=0.8)
fig.tight_layout()
fig.savefig(output_dir / 'frf.png', dpi=150)
print('Saved frf.png')

In [ ]:
# --- Display FRF inline ---
fig

In [ ]:
# --- Summary ---
print('=' * 50)
print('Example 05 — Geometric Nonlinearity Summary')
print('=' * 50)
print(f'  Linear natural frequency (mode 1) : {omega_linear:.4f} rad/s')
print(f'  Peak amplitude (DOF 0, harmonic 1): {peak_amp:.6f}')
print(f'  Frequency at peak                 : {peak_omega:.4f} rad/s')
print(f'  Hardening ratio  omega_peak/omega1: {hardening_ratio:.4f}')
print('=' * 50)